# Hybrid RAG Pipeline over AI Research Documents (`ai-docs`)

This notebook implements an end-to-end **Hybrid Retrieval-Augmented Generation (Hybrid RAG)** pipeline over the scientific research papers located in `./ai-docs/`.

### Architecture & Pipeline Overview:
1. **Document Ingestion**:
   - Uses LangChain's `PyPDFLoader` to load PDFs from `./ai-docs/`.
   - Employs `RecursiveCharacterTextSplitter` to create chunks with semantic overlap while preserving source file and page metadata.
2. **Dense Vector Embeddings**:
   - Google Vertex AI `text-embedding-004` (768 dimensions) captures semantic and conceptual meaning.
3. **Sparse Keyword Embeddings**:
   - `BM25Encoder` from `pinecone_text` computes term frequency / inverse document frequency (TF-IDF) sparse vectors for exact keyword matching (crucial for model names like `YOLOv11`, `VGG16`, `CNN`, medical terms like `NSCLC`, etc.).
4. **Pinecone Hybrid Search Retriever**:
   - Connected to Pinecone Serverless with `metric="dotproduct"`.
   - Combines dense and sparse vectors via convex combination $\alpha \cdot \text{dense} + (1 - \alpha) \cdot \text{sparse}$:
     - $\alpha = 1.0$: Pure dense (semantic) search.
     - $\alpha = 0.0$: Pure sparse (BM25 keyword) search.
     - $\alpha = 0.5$: Balanced hybrid search.
   - Uses a dedicated namespace `"ai-docs"` to maintain clean separation.
5. **Generative RAG Chain**:
   - LangChain Expression Language (LCEL) chain powered by Google Vertex AI `gemini-2.5-flash`.
   - Strictly grounded prompt enforcing source citations (document name and page number) and preventing hallucination.

In [1]:
# =====================================================================
# Step 1: Environment Setup & Imports
# =====================================================================
import os
import dotenv

# Load environment variables from .env (contains PINECONE_API)
dotenv.load_dotenv()

from pinecone import Pinecone, ServerlessSpec, Index
from pinecone_text.sparse import BM25Encoder
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

# Compatibility patch: LangChain Community passes vectors positionally to index.upsert,
# whereas the modern Pinecone client requires keyword-only arguments.
orig_upsert = Index.upsert
def patched_upsert(self, *args, **kwargs):
    if args:
        kwargs['vectors'] = args[0]
    return orig_upsert(self, **kwargs)
Index.upsert = patched_upsert

print("Environment setup and imports complete!")

C:\Users\Ritika Khandelwal\AppData\Local\Temp\ipykernel_26832\2410080609.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Environment setup and imports complete!


## Step 2: Initialize Google GenAI Embeddings

We initialize Google's `text-embedding-004` model via Vertex AI (`project="ai-api-2335"`) which outputs 768-dimensional dense vectors.

In [2]:
# =====================================================================
# Step 2: Google Vertex AI Embeddings (text-embedding-004)
# =====================================================================
embeddings = GoogleGenerativeAIEmbeddings(
    model="text-embedding-004",
    project="ai-api-2335",
    vertexai=True,
    region="global"
)

# Verify embeddings
sample_vector = embeddings.embed_query("Hybrid RAG pipeline")
print(f"Embedding model ready. Vector dimension: {len(sample_vector)}")

Embedding model ready. Vector dimension: 768


## Step 3: Connect to Pinecone & Setup Hybrid Index

Pinecone hybrid search requires the `dotproduct` similarity metric. We ensure the index exists, connect to it, and configure our target namespace (`"ai-docs"`).

In [3]:
# =====================================================================
# Step 3: Pinecone Client & Index Connection
# =====================================================================
pc_api_key = os.environ.get("PINECONE_API")
if not pc_api_key:
    raise ValueError("PINECONE_API not found in environment. Please check your .env file.")

pc = Pinecone(api_key=pc_api_key)
index_name = "hybrid-rag"
namespace = "ai-docs"

existing_indexes = pc.list_indexes().names()
print(f"Existing Pinecone indexes: {existing_indexes}")

# Note: Hybrid search requires metric="dotproduct" and dimension matching the embedding model (768)
if index_name not in existing_indexes:
    print(f"Creating index '{index_name}' with dotproduct metric for hybrid search...")
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)
stats = index.describe_index_stats()
print(f"Connected to Pinecone index: '{index_name}'")
print(f"Index stats: {stats}")

Existing Pinecone indexes: ['hybrid-rag', 'langchain-demo']
Connected to Pinecone index: 'hybrid-rag'
Index stats: DescribeIndexStatsResponse(dimension=768, total_vector_count=1199, metric='dotproduct', namespaces=2)


## Step 4: Load Documents from `ai-docs` with `PyPDFLoader`

We scan `./ai-docs/` and load all research papers page by page using `PyPDFLoader`.

In [52]:
# =====================================================================
# Step 4: Document Loading from ai-docs
# =====================================================================
docs_dir = "./ai-docs"
pdf_files = sorted([f for f in os.listdir(docs_dir) if f.endswith(".pdf")])
print(f"Found {len(pdf_files)} PDF documents in '{docs_dir}':\n")

all_docs = []
for idx, filename in enumerate(pdf_files, 1):
    file_path = os.path.join(docs_dir, filename)
    loader = PyPDFLoader(file_path)
    loaded_pages = loader.load()
    all_docs.extend(loaded_pages)
    print(f"  [{idx:02d}/{len(pdf_files):02d}] {filename} -> {len(loaded_pages)} pages")

print(f"\nTotal loaded pages across all documents: {len(all_docs)}")

Found 12 PDF documents in './ai-docs':

  [01/12] 1-s2.0-S1877050925029461-main.pdf -> 10 pages
  [02/12] 2404.03936v2.pdf -> 38 pages
  [03/12] 2510.26923v1.pdf -> 5 pages
  [04/12] 2602.10481v1.pdf -> 16 pages
  [05/12] AI-Powered Lung Cancer Detection- Assessing VGG16 and CNN Architectures for CT Scan Image Classification.pdf -> 30 pages
  [06/12] Assessment of Machine Learning Algorithms for Land Cover Classification in a Complex Mountainous Landscape.pdf -> 19 pages
  [07/12] Classi cation of NSCLC subtypes using lung microbiome from resected tissue based on machine learning methods.pdf -> 13 pages
  [08/12] Deep learning-based approach to diagnose lung cancer using CT-scan images.pdf -> 16 pages
  [09/12] Land-Cover Classification Using Deep Learning with High-Resolution Remote-Sensing Imagery.pdf -> 15 pages
  [10/12] LungPaper.pdf -> 25 pages
  [11/12] Utilizing Sentinel-2 Satellite Imagery for LULC and NDVI Change Dynamics for Gelephu Bhutan.pdf -> 21 pages
  [12/12] fmed-12-1

## Step 5: Document Chunking

We split the loaded pages into chunks using `RecursiveCharacterTextSplitter` (`chunk_size=1000`, `chunk_overlap=150`). Each chunk preserves metadata such as the source file path and page number.

In [53]:
# =====================================================================
# Step 5: Text Chunking with RecursiveCharacterTextSplitter
# =====================================================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(all_docs)
# Filter out any empty chunks
chunks = [c for c in chunks if c.page_content and c.page_content.strip()]

print(f"Total non-empty chunks created: {len(chunks)}")
print("\n--- Sample Chunk Preview ---")
print(f"Source: {chunks[0].metadata.get('source')} | Page: {chunks[0].metadata.get('page')}")
print(f"Content:\n{chunks[0].page_content[:300]}...")

Total non-empty chunks created: 1176

--- Sample Chunk Preview ---
Source: ./ai-docs\1-s2.0-S1877050925029461-main.pdf | Page: 0
Content:
ScienceDirect
Available online at www.sciencedirect.com
Procedia Computer Science 270 (2025) 1517–1526
1877-0509 © 2025 The Authors. Published by Elsevier B.V .
This is an open access article under the CC BY-NC-ND license (https://creativecommons.org/licenses/by-nc-nd/4.0)
Peer-review under responsi...


## Step 6: Sparse Encoder (BM25) Fitting & Persistence

`BM25Encoder` computes term frequency and inverse document frequency for exact keyword search. We fit the encoder on the corpus of chunks and persist it to `bm25_ai_docs.json`.

In [54]:
# =====================================================================
# Step 6: BM25 Sparse Encoder Setup & Persistence
# =====================================================================
bm25_path = "bm25_ai_docs.json"
encoder = BM25Encoder()

if os.path.exists(bm25_path):
    print(f"Loading saved BM25 encoder from '{bm25_path}'...")
    bm25 = encoder.load(bm25_path)
else:
    print("Fitting a new BM25 encoder on document chunks...")
    corpus = [chunk.page_content for chunk in chunks]
    bm25 = encoder
    bm25.fit(corpus)
    bm25.dump(bm25_path)
    print(f"BM25 encoder fitted and saved to '{bm25_path}'!")

print("BM25 encoder ready!")

Loading saved BM25 encoder from 'bm25_ai_docs.json'...
BM25 encoder ready!


## Step 7: Initialize Pinecone Hybrid Search Retriever

`PineconeHybridSearchRetriever` combines the dense embeddings from Google Vertex AI and sparse BM25 scores through convex scaling (`alpha * dense + (1 - alpha) * sparse`).

In [55]:
# =====================================================================
# Step 7: Initialize PineconeHybridSearchRetriever
# =====================================================================
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25,
    index=index,
    namespace=namespace,
    top_k=5,
    alpha=0.5  # 0.5 = Balanced hybrid (50% dense semantic, 50% sparse keyword)
)

print(f"Retriever ready! (namespace='{namespace}', top_k={retriever.top_k}, alpha={retriever.alpha})")

Retriever ready! (namespace='ai-docs', top_k=5, alpha=0.5)


## Step 8: Document Ingestion into Pinecone Hybrid Index

We check the vector count in the `ai-docs` namespace. If the namespace is not yet populated, we call `retriever.add_texts` to ingest the chunks with their metadata.

In [56]:
# =====================================================================
# Step 8: Ingest Documents into Hybrid Index
# =====================================================================
current_stats = index.describe_index_stats()
ns_info = current_stats.get('namespaces', {}).get(namespace, {})
ns_count = ns_info.get('vector_count', 0)
print(f"Current vector count in namespace '{namespace}': {ns_count}")

# Ingest if namespace is empty or if re-indexing is requested
reindex = False  # Set to True if you wish to overwrite/re-upload

if ns_count == 0 or reindex:
    print(f"Ingesting {len(chunks)} chunks into Pinecone namespace '{namespace}'...")
    texts = [c.page_content for c in chunks]
    metadatas = [c.metadata for c in chunks]
    retriever.add_texts(texts=texts, metadatas=metadatas, namespace=namespace)
    print("Ingestion complete!")
else:
    print(f"Namespace '{namespace}' already contains {ns_count} vectors. Skipping ingestion.")

Current vector count in namespace 'ai-docs': 1177
Namespace 'ai-docs' already contains 1177 vectors. Skipping ingestion.


## Step 9: Hybrid Search Retrieval Testing & Alpha Parameter Tuning

Here we test the retriever with domain queries from the research papers and inspect:
1. Retrieved chunks, relevance scores, and metadata.
2. Comparison between Pure Dense (`alpha=1.0`), Pure Sparse (`alpha=0.0`), and Balanced Hybrid (`alpha=0.5`).

In [57]:
# =====================================================================
# Step 9.1: Test Retrieval with Score and Metadata
# =====================================================================
query = "What deep learning architectures are evaluated for CT scan classification of lung cancer?"
print(f"Query: '{query}'\n" + "="*70)

retriever.alpha = 0.5  # Balanced hybrid
retriever.top_k = 4
results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    score = doc.metadata.get('score', 'N/A')
    score_str = f"{score:.4f}" if isinstance(score, float) else str(score)
    print(f"[{i}] Score: {score_str} | Source: {doc.metadata.get('source')} (Page {doc.metadata.get('page')})")
    print(f"Content snippet:\n{doc.page_content[:250]}...\n")

Query: 'What deep learning architectures are evaluated for CT scan classification of lung cancer?'
[1] Score: 0.6276 | Source: ai-docs\AI-Powered Lung Cancer Detection- Assessing VGG16 and CNN Architectures for CT Scan Image Classification.pdf (Page 1)
Content snippet:
cancer through CT scan image analysis using an openly accessible dataset for training and
evaluation. The model is trained to classify CT scan images into three categories: normal,
benign lung abnormalities, and malignant lung cancer, using a dataset...

[2] Score: 0.6243 | Source: ai-docs\AI-Powered Lung Cancer Detection- Assessing VGG16 and CNN Architectures for CT Scan Image Classification.pdf (Page 1)
Content snippet:
data, improving accuracy and consistency in  cancer detection [5]. Among various CNN 
architectures, Visual Geometry Group 16 (VGG16) has emerged as a leading model due 
to its deep layers and feature extracti on capabilities, making it highly e ﬀect...

[3] Score: 0.6158 | Source: ai-docs\Deep learning

In [58]:
# =====================================================================
# Step 9.2: Comparing Alpha Settings (Dense vs Sparse vs Hybrid)
# =====================================================================
keyword_query = "YOLOv11 lung nodule detection"
print(f"Query: '{keyword_query}'\n" + "="*70)

configs = [
    (1.0, "Dense Only (Semantic Search - text-embedding-004)"),
    (0.0, "Sparse Only (Keyword Search - BM25)"),
    (0.5, "Hybrid Search (Dense + Sparse Convex Combination)")
]

for a_val, desc in configs:
    retriever.alpha = a_val
    top_matches = retriever.invoke(keyword_query)
    top = top_matches[0] if top_matches else None
    print(f"\n[Alpha = {a_val}] {desc}:")
    if top:
        score = top.metadata.get('score', 'N/A')
        score_str = f"{score:.4f}" if isinstance(score, float) else str(score)
        print(f"Top Result Score: {score_str} | Source: {top.metadata.get('source')} (Page {top.metadata.get('page')})")
        print(f"Snippet: {top.page_content[:180]}...")

Query: 'YOLOv11 lung nodule detection'

[Alpha = 1.0] Dense Only (Semantic Search - text-embedding-004):
Top Result Score: 0.8467 | Source: ai-docs/2510.26923v1.pdf (Page 0)
Snippet: YOLOv11 is used for lung nodule detection in CT scan images....

[Alpha = 0.0] Sparse Only (Keyword Search - BM25):
Top Result Score: 0.7285 | Source: ai-docs/2510.26923v1.pdf (Page 0)
Snippet: YOLOv11 is used for lung nodule detection in CT scan images....

[Alpha = 0.5] Hybrid Search (Dense + Sparse Convex Combination):
Top Result Score: 0.7876 | Source: ai-docs/2510.26923v1.pdf (Page 0)
Snippet: YOLOv11 is used for lung nodule detection in CT scan images....


## Step 10: Build End-to-End Hybrid RAG Chain with Gemini & Source Citations

We assemble the full RAG chain using LangChain Expression Language (LCEL) with Google's `gemini-2.5-flash` LLM.

The prompt enforces:
1. Strict grounding in the retrieved context passages.
2. Explicit stating of "I don't know based on the provided documents" when information is absent.
3. Mandatory citation of the source document(s) and page number(s).

In [59]:
# =====================================================================
# Step 10: End-to-End RAG Chain with Gemini LLM
# =====================================================================
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    vertexai=True,
    project="ai-api-2335",
    location="us-central1",
    temperature=0
)

# Context-grounded citation prompt without terminal-breaking markdown formatting
prompt = ChatPromptTemplate.from_template("""
You are an expert AI research assistant. Answer the question using ONLY the provided context below.

Formatting & Output Rules:
1. Base your answer solely on the information provided in the Context passages.
2. If the answer is not present in the context, clearly state: "I don't know based on the provided documents."
3. Do NOT use Markdown formatting such as bold (**text** or __text__), italics (*text*), or markdown headers (###).
4. Output in clean, readable plain text suitable for terminal/CLI display. Use simple dashes (-) or numbers (1, 2, 3) for lists without any asterisks or bold formatting.
5. End your answer with a "Sources Cited:" section listing the document filename and page number for each source utilized, in plain text.

Context:
{context}

Question:
{question}

Answer:
""")

def format_docs_with_metadata(docs):
    formatted = []
    for doc in docs:
        src = doc.metadata.get("source", "Unknown Document")
        page = doc.metadata.get("page", "Unknown Page")
        formatted.append(f"[Document: {src} | Page: {page}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# Configure retriever for RAG
retriever.alpha = 0.5  # Balanced hybrid
retriever.top_k = 5

# Construct the LCEL chain
rag_chain = (
    {
        "context": retriever | format_docs_with_metadata,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Hybrid RAG Chain successfully assembled!")

Hybrid RAG Chain successfully assembled!


## Step 11: Execute Evaluation Queries over `ai-docs`

Let's test our Hybrid RAG pipeline with varied domain questions targeting the papers in `ai-docs`:

In [60]:
# =====================================================================
# Step 11: Execute Evaluation Queries with Streaming
# =====================================================================
def ask_rag_stream(question: str):
    print("=" * 80)
    print(f"QUESTION: {question}")
    print("=" * 80)
    print("\nANSWER:\n")
    for chunk in rag_chain.stream(question):
        print(chunk, end="", flush=True)
    print("\n\n" + "=" * 80 + "\n")

# Query 1: Lung Cancer detection with ML (Streaming + Plain Text, No Bold Formatting)
ask_rag_stream("How can lung cancer be detected with ML?")

# Query 2: Lung Cancer CT classification comparison
ask_rag_stream("How do VGG16 and CNN architectures compare for CT scan image classification of lung cancer?")

# Query 3: YOLOv11 data-efficient lung nodule detection
ask_rag_stream("What is the Scale-Aware Curriculum Learning approach for lung nodule detection with YOLOv11?")

# Query 4: Remote sensing & Satellite imagery analysis
ask_rag_stream("What deep learning approaches are discussed for satellite image time series analysis and land cover classification?")

# Query 5: Hallucination Prevention / Negative Test
ask_rag_stream("What was the stock price of Tesla in October 2026?")

QUESTION: How can lung cancer be detected with ML?

ANSWER:

Lung cancer can be detected with machine learning (ML) in several ways. Deep learning models, such as Convolutional Neural Networks (CNNs), can automatically learn relevant features from imaging data, like CT scans, to improve the accuracy and consistency of cancer detection.

Other machine learning approaches for lung cancer detection include:
- Using machine learning models based on handcrafted features.
- Employing Support Vector Machine (SVM) classifiers for early detection, often after pre-processing CT scan images, segmenting the Region of Interest (ROI), and compressing it.
- Utilizing feature extraction techniques like Local Energy-Based Shape Histogram (LESH).
- Applying soft computing techniques.
- Using enhanced artificial bee colony optimization techniques for detection and classification.

Sources Cited:
ai-docs\AI-Powered Lung Cancer Detection- Assessing VGG16 and CNN Architectures for CT Scan Image Classificati

In [61]:
ask_rag_stream('How does ML help detect lung cancer')

QUESTION: How does ML help detect lung cancer

ANSWER:

Machine learning (ML) helps detect lung cancer by offering significant advantages in medical image analysis. Deep learning models, such as Convolutional Neural Networks (CNNs), can automatically learn relevant features from imaging data, which improves accuracy and consistency in cancer detection. A CT-based object detection model for lung cancer can help medical professionals accurately locate lung cancer lesions, enhancing diagnostic precision. Such models can achieve high accuracy in predicting lung cancer types, supporting clinicians in making early diagnoses and developing timely treatment plans, which contributes to improved patient survival rates.

Sources Cited:
ai-docs\AI-Powered Lung Cancer Detection- Assessing VGG16 and CNN Architectures for CT Scan Image Classification.pdf | Page: 1
ai-docs\fmed-12-1567119.pdf | Page: 6




## Summary

- **Data Ingestion**: 12 academic PDF research papers in `ai-docs` loaded with `PyPDFLoader` and partitioned into granular text chunks with `RecursiveCharacterTextSplitter`.
- **Hybrid Indexing**: Combines Google Vertex AI `text-embedding-004` (768d dense) and `BM25Encoder` (sparse) within a Pinecone serverless `dotproduct` index under namespace `ai-docs`.
- **Retrieval**: `PineconeHybridSearchRetriever` delivers high-precision retrieval across both semantic concepts and specific model/scientific terminology.
- **Generation**: Google `gemini-2.5-flash` generates factual, context-grounded responses with direct citations to source PDFs and page numbers.